In [27]:
import numpy as np
import pandas as pd

# Load data
RAW_PATH = "ModeChoiceOptima.csv"
df_raw = pd.read_csv(RAW_PATH)
df=df_raw.copy()

In [28]:
# Column groups
ATTITUDE_COLS = [
    c for c in df_raw.columns
    if any(c.startswith(prefix) for prefix in ["Envir", "Mobil", "LifSty", "ResidCh"])
]

ENVIR_COLS  = [c for c in ATTITUDE_COLS if c.startswith("Envir")]
MOBIL_COLS  = [c for c in ATTITUDE_COLS if c.startswith("Mobil")]
LIFST_COLS  = [c for c in ATTITUDE_COLS if c.startswith("LifSty")]
RESID_COLS  = [c for c in ATTITUDE_COLS if c.startswith("ResidCh0")]

print(f"  Attitude columns : {len(ATTITUDE_COLS)}")
print(f"    Envir   : {len(ENVIR_COLS)}")
print(f"    Mobil   : {len(MOBIL_COLS)}")
print(f"    LifSty  : {len(LIFST_COLS)}")
print(f"    ResidCh : {len(RESID_COLS)}")
print()

CAT_COLS = ATTITUDE_COLS + [
    "Gender","HouseType","OwnHouse","Internet","NewsPaperSubs",
    "Mothertongue","FamilSitu","OccupStat","SocioProfCat","Education",
    "CarAvail","Income","TripPurpose","DestAct","ModeToSchool",
    "ResidChild","FreqCarPar","FreqTrainPar","FreqOtherPar",
    "FreqTripHouseh","HalfFareST","LineRelST","GenAbST",
    "AreaRelST","OtherST",
]

CONT_COLS = [
    "age","CalculatedIncome","NbHousehold","NbChild","NbCar","NbMoto",
    "NbBicy","NbBicyChild","NbComp","NbTV","NbCellPhones","NbSmartPhone",
    "NbRoomsHouse","YearsInHouse","BirthYear","ReportedDuration","InVehicleTime",
]

  Attitude columns : 55
    Envir   : 6
    Mobil   : 27
    LifSty  : 14
    ResidCh : 7



In [29]:
# Step 1: drop missing target
#Drop the rows with -1 values in the Choice column
df.drop(df[df['Choice'] == -1].index, inplace=True)
#Checking the count of entries after removal of -1 in the Choice column
df.count()

ID                  1906
DestAct             1906
NbTransf            1906
TimePT              1906
WalkingTimePT       1906
                    ... 
ModeToSchool        1906
ReportedDuration    1906
CoderegionCAR       1906
age                 1906
Weight              1906
Length: 117, dtype: int64

In [30]:
# Step 2: drop rows with -2 values
df = df[(df!= -2).all(axis=1)]
df.count()

ID                  1871
DestAct             1871
NbTransf            1871
TimePT              1871
WalkingTimePT       1871
                    ... 
ModeToSchool        1871
ReportedDuration    1871
CoderegionCAR       1871
age                 1871
Weight              1871
Length: 117, dtype: int64

In [31]:
#Step 3: Convert -1 values to NaN
df.replace(-1, np.nan, inplace=True)
df.count()

ID                  1871
DestAct             1858
NbTransf            1871
TimePT              1871
WalkingTimePT       1871
                    ... 
ModeToSchool        1775
ReportedDuration    1804
CoderegionCAR       1871
age                 1789
Weight              1871
Length: 117, dtype: int64

In [32]:
# Step 4: mode imputation
print("=" * 60)
print("STEP 4 — Mode imputation")
print("=" * 60)

n_imputed_step4 = 0
for col in CAT_COLS:
    if col not in df.columns:
        continue
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        df[col] = df[col].fillna(df[col].mode()[0])
        n_imputed_step4 += n_missing

print(f"  Cells imputed : {n_imputed_step4:,}")
print()


STEP 4 — Mode imputation
  Cells imputed : 4,639



In [33]:
# Step 5: median imputation
print("=" * 60)
print("STEP 5 — Median imputation")
print("=" * 60)

n_imputed_step5 = 0
for col in CONT_COLS:
    if col not in df.columns:
        continue
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        df[col] = df[col].fillna(df[col].median())
        n_imputed_step5 += n_missing

print(f"  Cells imputed : {n_imputed_step5:,}")
print()


STEP 5 — Median imputation
  Cells imputed : 1,648



In [34]:
# Final check
total_nan_final = df.isnull().sum().sum()

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"  Original shape : {df_raw.shape}")
print(f"  Final shape    : {df.shape}")
print(f"  Remaining NaN  : {total_nan_final}")
print()

print("  Final Choice distribution:")
print(df["Choice"].value_counts().sort_index().to_string())


FINAL SUMMARY
  Original shape : (2265, 117)
  Final shape    : (1871, 117)
  Remaining NaN  : 0

  Final Choice distribution:
Choice
0     525
1    1233
2     113


In [35]:
# Save
OUTPUT_PATH = "ModeChoiceOptima_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)

print()
print(f"Saved to: {OUTPUT_PATH}")


Saved to: ModeChoiceOptima_cleaned.csv


In [39]:
df_stat=df.describe()
df_stat.to_csv("ModeChoiceOptima_cleaned_stats.csv")